# Business Analytics — Olist Brazilian E-Commerce Dataset
## Phase 1: Business Context & Dataset Overview
## Phase 2: Data Profiling & Exploratory Data Analysis

---


# PHASE 1 — Business Context & Dataset Overview


## 1.1 Business Domain & Context

**Olist** is a Brazilian e-commerce marketplace intermediary that connects small and medium-sized merchants to major online retail channels across Brazil. Operating like a 'store of stores', Olist allows independent sellers to list their products on its platform and fulfills orders through logistics partners, handling payments and customer communications on behalf of sellers.

The dataset captures the complete order lifecycle — from the moment a customer places an order to product delivery and post-purchase review — across multiple Brazilian states between **September 2016 and September 2018**. The business domain spans **e-commerce operations, supply chain logistics, customer experience management, and retail analytics**.

Key stakeholders who benefit from analysing this data include:
- **Operations teams** – monitoring delivery performance and logistics efficiency
- **Product managers** – identifying top-selling categories and pricing strategies
- **Customer experience teams** – understanding satisfaction drivers through reviews
- **Finance teams** – analysing revenue, payment behaviour, and freight cost structures


## 1.2 Dataset Overview

| Attribute | Detail |
|-----------|--------|
| **Source** | Kaggle – *Brazilian E-Commerce Public Dataset by Olist* (originally released by Olist Store) |
| **Original Format** | 9 relational CSV files (customers, orders, order items, payments, reviews, products, sellers, geolocation, category translation) |
| **Working Format** | Single consolidated `.xlsx` / `.csv` — joined on `order_id`, `product_id`, `seller_id`, `customer_id` |
| **Number of Records** | **112,650 rows** (order line items) |
| **Number of Attributes** | **28 columns** |
| **Time Period** | September 2016 – September 2018 |
| **Geographic Scope** | Brazil (27 states + Federal District) |


## 1.3 Data Dictionary

| # | Column Name | Data Type | Description |
|---|-------------|-----------|-------------|
| 1 | `Order ID` | String (ID) | Unique identifier for each order |
| 2 | `Order Status` | Categorical | Order lifecycle status: *delivered, shipped, canceled, invoiced, processing, unavailable, approved* |
| 3 | `Order Date` | Datetime | Timestamp when the customer placed the order |
| 4 | `Approved Date of order` | Datetime | Timestamp when payment was approved |
| 5 | `Carrier Pickup Date` | Datetime | Timestamp when the logistics carrier collected the package |
| 6 | `Delivery Date` | Datetime | Timestamp when the package was delivered to the customer |
| 7 | `Estimated Delivery Date` | Datetime | System-estimated delivery date shown to the customer at purchase |
| 8 | `Number of Item/s` | Integer | Sequential item counter within a single order (1 = first item, 2 = second, etc.) |
| 9 | `Product ID` | String (ID) | Unique product identifier |
| 10 | `Product Category` | Categorical | English product category (e.g., *health_beauty*, *bed_bath_table*) |
| 11 | `Product Weight (g)` | Float | Product weight in grams |
| 12 | `Product Length(cm)` | Float | Product length in centimetres |
| 13 | `Product Height(cm)` | Float | Product height in centimetres |
| 14 | `Product Width (cm)` | Float | Product width in centimetres |
| 15 | `Price of Item` | Float (BRL) | Listed price of the product in Brazilian Reais |
| 16 | `Freight Cost` | Float (BRL) | Shipping/freight cost charged to the customer |
| 17 | `Payment Type` | Categorical | Payment method used: *credit_card, boleto, voucher, debit_card* |
| 18 | `payment_installments` | Float | Number of installments used by the customer for payment |
| 19 | `Payment Value` | Float (BRL) | Total payment amount for the order |
| 20 | `review_score` | Float (1–5) | Customer satisfaction score left after delivery |
| 21 | `review_comment_message` | String | Optional free-text review comment (mostly in Portuguese) |
| 22 | `seller_id` | String (ID) | Unique identifier for the seller |
| 23 | `seller_city` | String | City where the seller is located |
| 24 | `seller_state` | Categorical | Brazilian state code of the seller (e.g., SP, RJ, MG) |
| 25 | `seller_zip_code` | Integer | Seller's zip code prefix |
| 26 | `customer_city` | String | City where the customer is located |
| 27 | `customer_state` | Categorical | Brazilian state code of the customer |
| 28 | `shipping_deadline` | Datetime | Seller's deadline to hand off the package to the carrier |


## 1.4 Business Questions

### Standard Questions (Q1–Q5)

**Q1.** Which product categories generate the highest total revenue, and how does their average item price compare?  
*(Insight: identifies high-value product lines for marketing and inventory prioritization)*

**Q2.** What is the distribution of delivery times (in days), and how often are orders delivered before, on, or after the estimated delivery date?  
*(Insight: measures logistics performance and customer expectation fulfilment)*

**Q3.** How does payment method (credit card, boleto, voucher, debit card) vary across order value segments?  
*(Insight: reveals customer payment behaviour and financial risk by segment)*

**Q4.** What is the relationship between freight cost and product weight/dimensions — and which seller states charge the highest average freight?  
*(Insight: supports freight pricing model review and regional logistics cost analysis)*

**Q5.** What is the average review score per product category, and are there categories with consistently low satisfaction?  
*(Insight: pinpoints categories requiring seller quality control or customer communication improvements)*

### Ad Hoc Questions (Q6–Q7)

**Q6.** How has monthly order volume and revenue trended over the dataset's two-year period — and are there visible seasonal peaks?  
*(Insight: supports demand forecasting, campaign planning, and capacity management)*

**Q7.** Which sellers (by state) account for the majority of orders, and is there a concentration risk (i.e., a few sellers generating most of the volume)?  
*(Insight: highlights geographic seller concentration and platform dependency risk)*


---
# PHASE 2 — Data Profiling & Exploratory Data Analysis


## 2.1 Setup & Data Loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Styling ──────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.titlesize': 15
})

# ── Load data ────────────────────────────────────────────────
df = pd.read_excel('Business_ANalytics.xlsx')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


## 2.2 Initial Inspection


In [ ]:
# Shape
print("Shape:", df.shape)
print()

# Column types
print("Data Types:")
print(df.dtypes.to_string())


In [ ]:
# .info() summary
df.info()


In [ ]:
# Numeric descriptive statistics
numeric_cols = [
    'Product Weight (g)', 'Product Length(cm)', 'Product Height(cm)',
    'Product Width (cm)', 'Price of Item', 'Freight Cost',
    'payment_installments', 'Payment Value', 'review_score', 'Number of Item/s'
]
df[numeric_cols].describe().round(2)


## 2.3 Missing Value Analysis & Treatment


In [ ]:
# Count and percentage of missing values per column
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)
print(missing.to_string())


In [ ]:
# ── Visualise missingness ─────────────────────────────────────
missing_viz = missing.reset_index().rename(columns={'index': 'Column'})

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing_viz['Column'], missing_viz['Missing %'], color='#e07b54', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Rate by Column', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# ── Treatment strategy ────────────────────────────────────────
df_clean = df.copy()

# 1. Carrier Pickup Date & Delivery Date: missing for non-delivered orders — FLAG, do not impute
df_clean['delivery_missing_flag'] = df_clean['Delivery Date'].isnull().astype(int)

# 2. Product dimensions/weight: 18 rows (<0.02%) — drop rows
df_clean = df_clean.dropna(subset=['Product Weight (g)', 'Product Length(cm)',
                                    'Product Height(cm)', 'Product Width (cm)'])

# 3. Payment fields: 3 rows — drop rows
df_clean = df_clean.dropna(subset=['Payment Type', 'payment_installments', 'Payment Value'])

# 4. review_score: 942 rows (~0.8%) — impute with median (customer didn't leave a score)
median_score = df_clean['review_score'].median()
df_clean['review_score'] = df_clean['review_score'].fillna(median_score)

# 5. Product Category: 1,627 rows — flag as 'Unknown'
df_clean['Product Category'] = df_clean['Product Category'].fillna('Unknown')

# 6. review_comment_message: ~58% missing — expected (optional field), leave as-is
# 7. Approved/Carrier dates: small % for logistics analysis, keep with flag

print(f"Rows after cleaning: {df_clean.shape[0]:,} (removed {len(df) - df_clean.shape[0]} rows)")
print(f"Remaining nulls:\n{df_clean.isnull().sum()[df_clean.isnull().sum() > 0].to_string()}")


## 2.4 Outlier Detection (IQR Method)


In [ ]:
# IQR-based outlier detection
outlier_cols = ['Price of Item', 'Freight Cost', 'Payment Value',
                'Product Weight (g)', 'review_score']

results = []
for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    results.append({
        'Column': col, 'Q1': round(Q1,2), 'Q3': round(Q3,2),
        'IQR': round(IQR,2), 'Lower Fence': round(lower,2),
        'Upper Fence': round(upper,2), 'Outliers (n)': n_out,
        'Outliers (%)': round(n_out / len(df_clean) * 100, 2)
    })

outlier_df = pd.DataFrame(results).set_index('Column')
print(outlier_df.to_string())


In [ ]:
# ── Box plots for outlier visualisation ──────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, col in zip(axes, outlier_cols):
    ax.boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#5b9bd5', alpha=0.7),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(col.replace(' (g)','\n(g)').replace('(',\'\n('), fontsize=9)
    ax.set_xlabel('')

fig.suptitle('Outlier Detection — Box Plots (IQR Method)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nNote: Outliers are retained in the dataset as they represent genuine high-value orders")
print("and heavy products, not data entry errors. Flagging is sufficient for modelling purposes.")


## 2.5 Distribution Analysis (Histograms & Box Plots)


In [ ]:
# ── Histograms for all numeric variables ─────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df_clean[col].dropna()
    axes[i].hist(data, bins=40, color='#5b9bd5', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Distribution of All Numeric Variables', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── Price of Item: log scale for skewed dist ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_clean['Price of Item'], bins=60, color='#e07b54', edgecolor='white', alpha=0.85)
axes[0].set_title('Price of Item — Original Scale')
axes[0].set_xlabel('Price (BRL)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df_clean['Price of Item']), bins=60, color='#70ad47', edgecolor='white', alpha=0.85)
axes[1].set_title('Price of Item — Log Scale (log1p)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

fig.suptitle('Item Price Distribution', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Review score distribution (discrete) ─────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
score_counts = df_clean['review_score'].value_counts().sort_index()
bars = ax.bar(score_counts.index.astype(int), score_counts.values,
              color=['#e07b54','#f4a56a','#ffd966','#9dc3e6','#4472c4'],
              edgecolor='white', width=0.6)
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=9)
ax.set_xlabel('Review Score')
ax.set_ylabel('Number of Orders')
ax.set_title('Review Score Distribution (1 = Worst, 5 = Best)', fontweight='bold')
ax.set_xticks([1,2,3,4,5])
plt.tight_layout()
plt.show()


In [ ]:
# ── Payment type distribution ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
pt = df_clean['Payment Type'].value_counts()
wedges, texts, autotexts = ax.pie(
    pt.values, labels=pt.index, autopct='%1.1f%%',
    colors=['#4472c4','#e07b54','#a9d18e','#ffd966'],
    startangle=140, pctdistance=0.8
)
ax.set_title('Payment Type Distribution', fontweight='bold')
plt.tight_layout()
plt.show()


## 2.6 Correlation Matrix


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
corr = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle only (show lower)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax, vmin=-1, vmax=1,
    annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix — Numeric Variables', fontweight='bold', pad=15)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Top positive correlations ─────────────────────────────────
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'Var1', 'level_1': 'Var2', 0: 'Correlation'})
    .sort_values('Correlation', key=abs, ascending=False)
)
print("Top 10 Correlations (by absolute value):")
print(corr_pairs.head(10).to_string(index=False))


## 2.7 EDA Summary Narrative


In [ ]:
# ── Delivery performance summary ──────────────────────────────
delivered = df_clean[df_clean['Order Status'] == 'delivered'].copy()
delivered['delivery_days'] = (delivered['Delivery Date'] - delivered['Order Date']).dt.days
delivered['days_vs_estimate'] = (delivered['Delivery Date'] - delivered['Estimated Delivery Date']).dt.days

early = (delivered['days_vs_estimate'] < 0).sum()
on_time = (delivered['days_vs_estimate'] == 0).sum()
late = (delivered['days_vs_estimate'] > 0).sum()
total_d = len(delivered)

print("=== Delivery Performance ===")
print(f"Average delivery time  : {delivered['delivery_days'].mean():.1f} days (median: {delivered['delivery_days'].median():.0f})")
print(f"Delivered early        : {early:,} ({early/total_d*100:.1f}%)")
print(f"Delivered on exact date: {on_time:,} ({on_time/total_d*100:.1f}%)")
print(f"Delivered late         : {late:,} ({late/total_d*100:.1f}%)")

print("\n=== Revenue Summary ===")
print(f"Total Payment Value    : BRL {df_clean['Payment Value'].sum():,.0f}")
print(f"Avg order value        : BRL {df_clean['Payment Value'].mean():,.2f}")
print(f"Avg item price         : BRL {df_clean['Price of Item'].mean():,.2f}")
print(f"Avg freight cost       : BRL {df_clean['Freight Cost'].mean():,.2f}")
print(f"Freight-to-price ratio : {df_clean['Freight Cost'].mean()/df_clean['Price of Item'].mean()*100:.1f}%")

print("\n=== Review Score Summary ===")
print(f"Average review score   : {df_clean['review_score'].mean():.2f} / 5.0")
print(f"% Score 5              : {(df_clean['review_score']==5).mean()*100:.1f}%")
print(f"% Score 1 or 2         : {(df_clean['review_score']<=2).mean()*100:.1f}%")

print("\n=== Credit Usage ===")
cc = df_clean[df_clean['Payment Type']=='credit_card']
print(f"Credit card orders     : {len(cc):,} ({len(cc)/len(df_clean)*100:.1f}%)")
print(f"Avg installments (CC)  : {cc['payment_installments'].mean():.1f}")


### Key Observations from EDA

**Observation 1 — Price and payment distributions are heavily right-skewed.**  
The median item price is BRL 75, but the mean is BRL 121 and the maximum reaches BRL 6,735. This skew is driven by a small number of high-value electronics, computer, and appliance orders. Log transformation is recommended for any price-based modelling. Similarly, freight cost distributions show 10.8% of orders are IQR-classified outliers, correlating strongly with product weight (r = 0.61).

**Observation 2 — Logistics performance is mostly positive but with notable late deliveries.**  
The average delivery time is 12 days, and approximately 77% of delivered orders arrive before the estimated delivery date shown to customers. However, ~20% of delivered orders arrive late, with extreme cases reaching 209 days. This tail is a significant customer satisfaction risk, as review scores drop sharply for delayed orders.

**Observation 3 — Review scores are heavily polarised towards 5 stars.**  
Over 57% of all scored orders receive a perfect 5-star rating, while ~11% score 1 or 2 stars. This U-shaped distribution (many 5s and 1s, fewer 2–4s) is characteristic of marketplace review systems where satisfied customers and very dissatisfied customers are most motivated to leave feedback. The weak negative correlation between review score and payment value (r = −0.08) suggests higher-priced purchases attract slightly more critical reviews.

**Observation 4 — Credit card dominates payment, with meaningful installment usage.**  
76.3% of orders are paid by credit card, and the average number of installments among credit card users is approximately 3. This reflects Brazil's widely used parcelamento (installment payment) culture, which allows consumers to spread purchases over months. Boleto (a bank slip payment method) accounts for 20.3% of orders — important for reaching Brazil's unbanked population.

**Observation 5 — Product weight and freight cost are the strongest correlated pair (r = 0.61).**  
Among all numeric variable pairs, product weight and freight cost show the highest correlation, confirming that logistics pricing is largely weight-driven. Product dimensions (length, height, width) also correlate moderately with freight (0.31–0.39), consistent with volumetric pricing models. Notably, payment value and price of item are highly correlated (r = 0.76), as expected since the total order value is primarily driven by item price.
